<a href="https://colab.research.google.com/github/britoadriana/ScreenQA-PT-Eval/blob/main/Notebooks/GIT_EXAMPLE_Qwen_ScreenQA_PT_Analysis_LLM_as_a_judge_comented.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Qwen**

Model: https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

In [ ]:
!pip install qwen-vl-utils[decord] transformers

In [ ]:
!pip install datasets

In [ ]:
!pip install -q langchain-groq langchain-core pydantic

**Imports**

In [ ]:
import matplotlib.pyplot as plt
import os
import requests
from PIL import Image
import torch
from transformers import BitsAndBytesConfig
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
from datasets import load_dataset
from typing import Optional
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
import os
from google.colab import userdata
from getpass import getpass
import time
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

In [ ]:
MODEL_CACHE_DIR = r"/content/drive/MyDrive/qwen"
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
# Criar pasta se não existir
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)

# Variaveis que o HF usa para guardar cache, substituindo pelo meu diretório
os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = MODEL_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = MODEL_CACHE_DIR

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
     model_id,
     cache_dir=MODEL_CACHE_DIR,
     dtype="auto",
     device_map="auto",
)

processor = AutoProcessor.from_pretrained(
      model_id,
      cache_dir=MODEL_CACHE_DIR,
)

print("Model loaded!")
print("Model ID:", model_id)

**Model testing**

In [ ]:
conversation = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "http://images.cocodataset.org/val2017/000000039769.jpg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    conversation, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(conversation)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

In [ ]:
image_file = "http://images.cocodataset.org/val2017/000000039769.jpg"
raw_image = Image.open(requests.get(image_file, stream=True).raw)
plt.imshow(raw_image)
plt.axis('off')
plt.show()

**Dataset**

In [ ]:
dataset = load_dataset(
    "britoadriana/screen_qa_portuguese",
    split="test",
    cache_dir="/content/drive/MyDrive/screen_qa_portuguese"
)

print(dataset)

In [ ]:
item = dataset[1]
screen_id = item["resource_id"]
question_file = item["question"]
image_file = item["image"]
resposta_file = item["answers"]

print("Screen id:", item["resource_id"])
print("Question:", item["question"])
print("Answers:", item["answers"])
print("Image_file:", item["image"])

plt.imshow(item["image"])
plt.axis("off")
plt.show()

**Performing model inference across the entire dataset and exporting the outputs to JSON format to facilitate downstream analysis**

In [ ]:
# Configuration and Directory Setup
# Define the Google Drive directory and the final output filename
DRIVE_FOLDER = "/content/drive/MyDrive/results_my_dataset_qwen_screenqa_json"
FILENAME = "results_my_dataset_qwen_screenqa.json"

# Create the directory if it doesn't exist
os.makedirs(DRIVE_FOLDER, exist_ok=True)
OUTPUT_FILE = os.path.join(DRIVE_FOLDER, FILENAME)

# Checkpoint Management
final_results = []
partial_name = f"partial_{FILENAME}"
partial_path = os.path.join(DRIVE_FOLDER, partial_name)

initial_index = 0

# Check for existing partial progress to enable resuming
if os.path.exists(partial_path):
    print("Partial file found. Loading...")
    with open(partial_path, "r", encoding="utf-8") as f:
        final_results = json.load(f)

    # Resume from the index following the last saved sequential_id
    initial_index = final_results[-1]["sequential_id"] + 1
    print(f"Resuming from item... {initial_index}")
else:
    print("No partial file found. Starting from scratch.")

# Inference Loop
print(f"Initiating batch inference. Saving to: {OUTPUT_FILE}")

for i in tqdm(
    range(initial_index, len(dataset)),
    total=len(dataset) - initial_index
):
    item = dataset[i]

    # Data Extraction
    question = item["question"]
    raw_image = item["image"]
    ground_truth = item["answers"]

    # Extract or generate a unique identifier for the image
    image_identifier = getattr(raw_image, 'filename', f"example_{i}")
    screen_title = os.path.basename(image_identifier)

    # Construct the multimodal conversation template
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": f"{question} Be concise."},
                {"type": "image"},
            ],
        },
    ]

    # Model Inference Execution
    try:
        text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(
            images=raw_image,
            text=text_prompt,
            return_tensors='pt'
        ).to(model.device, torch.float16)

        output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

        # Calculate input size to slice only the newly generated tokens
        input_size = inputs.input_ids.shape[1]
        generated_ids = output_ids[0][input_size:]

        text_response = processor.decode(generated_ids, skip_special_tokens=True).strip()
    except Exception as e:
        print(f"Error at item {i}: {e}")
        text_response = "GENERATION_ERROR"

    # Data Persistence Object
    result_entry = {
        "sequential_id": i,
        "screen_title": screen_title,
        "question": question,
        "ground_truth": ground_truth,
        "model_prediction": text_response,
        "resource_id_element": item["resource_id"]
    }

    final_results.append(result_entry)

    # Save a checkpoint every 100 items to prevent data loss
    if i % 100 == 0:
        with open(partial_path, 'w', encoding='utf-8') as f:
            json.dump(final_results, f, ensure_ascii=False, indent=2)

# Final Export
# Save the complete results to the main output file
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print(f"Completed! Results saved to: {OUTPUT_FILE}")

**LLM-as-a-judge for semantic validation**

Since model responses may be semantically equivalent to the ground truth despite different phrasing, keyword-based metrics are inherently limited. Therefore, employing a high-capacity LLM as a judge provides a more accurate assessment. While subject to potential hallucinations, this approach offers greater precision than traditional lexical metrics in evaluating semantic adequacy.

**gpt-oss-120b**

In [ ]:
try:
    secret_key = userdata.get('GROQ_API_KEY')
    os.environ["GROQ_API_KEY"] = secret_key
    print("API Key loaded successfully from Colab Secrets.")

except Exception:
    if "GROQ_API_KEY" not in os.environ:
        print("Key not found in Secrets. Enter your API Key below:")
        os.environ["GROQ_API_KEY"] = getpass()

# Output Structure Definition (Strict JSON Format)
class JudgeVerdict(BaseModel):
    score: int = Field(
        description="Score from 0 to 10 evaluating the prediction accuracy against the ground truth."
    )
    justification: str = Field(
        description="Short explanation of the reasoning behind the score."
    )

# Model Initialization
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0, # Zero Temperature for Maximum Objectivity
)

# Enforcing Structured Output
structured_evaluator = llm.with_structured_output(JudgeVerdict)

#The Judge Prompt
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an AI judge specialized in ScreenQA (User Interface Question Answering).
    Your task is to assign a score from 0 to 10 by comparing the PREDICTION with the GROUND TRUTH.

    Scoring Criteria:
    - 10: Perfect. Exactly the same or a perfect synonym.
    - 8-9: Correct, but with irrelevant variations in format (e.g., '12' vs 'twelve').
    - 7: Correct semantic core. The response correctly conveys the primary information requested, although it may contain minor omissions, simplifications, or redundant information that does not compromise the central meaning.
    - 5–6: Partially correct. Captures only part of the primary information or presents omissions/additions that affect the completeness or accuracy of the response.
    - 3-4: Incorrect. Fails to provide the primary information but demonstrates some contextual understanding or mentions related elements.
    - 0-2: Completely incorrect, irrelevant, or total hallucination.

    Respond strictly in the JSON format below:
    {format_instructions}
    """),
    ("human", """
    QUESTION: {question}
    GROUND TRUTH: {ground_truth}
    PREDICTION: {prediction}
    """)
])

# Adjusted Parser for the Updated Data Object
parser = JsonOutputParser(pydantic_object=JudgeVerdict)

# Chain Construction
prompt_with_instructions = judge_prompt.partial(format_instructions=parser.get_format_instructions())
evaluation_chain = prompt_with_instructions | llm | parser

def extract_ground_truth_text(raw_ground_truth):
    """Helper function to handle different ground_truth formats."""
    try:
        if isinstance(raw_ground_truth, list):
            return raw_ground_truth[0].get('full_answer', str(raw_ground_truth[0]))
        elif isinstance(raw_ground_truth, dict):
            return raw_ground_truth.get('full_answer', str(raw_ground_truth))
        return str(raw_ground_truth)
    except Exception:
        return str(raw_ground_truth)

def run_evaluation(input_path, output_path):
    print(f"Reading file: {input_path}")

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    evaluated_results = []
    api_errors = 0
    total_items = len(data)

    print(f"Starting evaluation of {total_items} items...")

    for item in tqdm(data):
        raw_ground_truth = item.get("ground_truth", "")
        ground_truth_text = extract_ground_truth_text(raw_ground_truth)

        prediction = item.get("model_prediction", "")
        question = item.get("question", "")

        # Skip evaluation if previous generation failed
        if not prediction or prediction == "GENERATION_ERROR":
            item["judge_evaluation"] = {"score": 0, "reason": "Previous generation failure"}
            evaluated_results.append(item)
            continue

        try:
            # Call LLM Judge
            result: JudgeVerdict = evaluation_chain.invoke({
                "question": question,
                "ground_truth": ground_truth_text,
                "prediction": prediction
            })

            item["judge_evaluation"] = {
                "score": result["score"],
                "reason": result["justification"]
            }

        except Exception as e:
            item["judge_evaluation"] = {"score": 0, "reason": f"API Error: {str(e)}"}
            api_errors += 1

        evaluated_results.append(item)

        # Partial saving every 50 items
        if len(evaluated_results) % 50 == 0:
             with open(output_path, "w", encoding="utf-8") as f:
                json.dump(evaluated_results, f, ensure_ascii=False, indent=2)

    # Final Data Persistence
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(evaluated_results, f, ensure_ascii=False, indent=2)

    print("\n" + "="*40)
    print("PROCESSING COMPLETED")
    print(f"Total Processed: {total_items}")
    print(f"API Errors: {api_errors}")
    print(f"File saved to: {output_path}")
    print("="*40)

In [ ]:
DRIVE_FOLDER = "/content/drive/MyDrive/resultados_meu_dataset_qwen_screenqa_json"
FILENAME = "resultados_meu_dataset_qwen_screenqa.json"

os.makedirs(DRIVE_FOLDER, exist_ok=True)
OUTPUT_FILE = os.path.join(DRIVE_FOLDER, FILENAME)


OUTPUT_FILE_NAME = "resultados_meru_dataset_avaliado_GPT_qwen.json"

input_path = OUTPUT_FILE
output_path = os.path.join(DRIVE_FOLDER, OUTPUT_FILE_NOME)

In [ ]:
if os.path.exists(input_path):
    rodar_avaliacao(input_path, output_path)
else:
    print(f"ERROR:\n{input_path}")

**SQUAD F1 SQUAD M COSSENO**

In [ ]:
import pandas as pd
import json
import re
import string
import os
import unicodedata
from collections import Counter
from sentence_transformers import SentenceTransformer, util

def remove_accents(text):
    """Removes accents: essential for Portuguese (e.g., 'botão' becomes 'botao')"""
    return ''.join(ch for ch in unicodedata.normalize('NFKD', text)
                  if unicodedata.category(ch) != 'Mn')

def normalize_text(text):
    if not text: return ""
    text = str(text).lower()

    # Normalize accents to avoid penalizing the model for missing diacritics
    text = remove_accents(text)

    # Portuguese Stopword Removal: Stripping articles (o, a, os, as, um, uma, uns, umas).
    text = re.sub(r'\b(o|a|os|as|um|uma|uns|umas)\b', ' ', text)

    exclude = set(string.punctuation)
    text = ''.join(ch for ch in text if ch not in exclude)
    return ' '.join(text.split())

def compute_metrics(gold_answer, prediction):
    norm_gold = normalize_text(gold_answer)
    norm_pred = normalize_text(prediction)

    # Exact Match (EM)
    em = 1 if norm_gold == norm_pred else 0

    gold_toks = norm_gold.split()
    pred_toks = norm_pred.split()

    if len(gold_toks) == 0 or len(pred_toks) == 0:
        return em, (1 if gold_toks == pred_toks else 0)

    common = Counter(gold_toks) & Counter(pred_toks)
    num_same = sum(common.values())

    if num_same == 0: return em, 0

    precision = 1.0 * num_same / len(pred_toks)
    recall = 1.0 * num_same / len(gold_toks)
    f1 = (2 * precision * recall) / (precision + recall)

    return em, f1

# Initialize Embedding Model
embedding_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

def calculate_semantic_similarity(text1, text2):
    """Calculates semantic similarity (cosine) regardless of identical wording."""
    if not text1 or not text2: return 0.0

    emb1 = embedding_model.encode(text1, convert_to_tensor=True)
    emb2 = embedding_model.encode(text2, convert_to_tensor=True)

    cosine_scores = util.cos_sim(emb1, emb2)
    return cosine_scores.item()

# Data Processing Pipeline

input_json_path = "/content/drive/MyDrive/results_my_dataset_qwen_screenqa_json/results_my_dataset_evaluated_GPT_qwen.json"
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

if os.path.exists(input_json_path):
    with open(input_json_path, 'r', encoding='utf-8') as f:
        data_content = json.load(f)
else:
    print(f"Error: File not found at {input_json_path}")
    data_content = []

rows = []

for item in data_content:
    prediction = item.get("model_prediction", "")
    ground_truths = item.get("ground_truth", [])

    max_em, max_f1, max_cosine = 0, 0, 0

    # Ensure ground truths are formatted as lists (SQuAD standard)
    if not isinstance(ground_truths, list):
        ground_truths = [ground_truths]

    if len(ground_truths) > 0:
        for gt in ground_truths:
            # SQuAD Metrics (Lexical / Word-level)
            current_em, current_f1 = compute_metrics(gt, prediction)
            max_em = max(max_em, current_em)
            max_f1 = max(max_f1, current_f1)

            # Semantic Similarity (Meaning-based)
            current_cos = calculate_semantic_similarity(gt, prediction)
            max_cosine = max(max_cosine, current_cos)

        gt_display = ground_truths[0]
    else:
        gt_display = ""

    # LLM as a judge metrics
    judge_evaluation = item.get("judge_evaluation", {})
    score_val = judge_evaluation.get("score")

    try:
        score_num = int(score_val) if score_val is not None else 0
    except:
        score_num = 0

    rows.append({
        "id": item.get("sequential_id"),
        "screen_title": item.get("screen_title"),
        "question": item.get("question"),
        "ground_truth": gt_display,
        "model_prediction": prediction,
        "squad_f1": round(max_f1, 4),
        "squad_em": max_em,
        "semantic_similarity": round(max_cosine, 4),
        "gpt_judge_score": score_num,
        "judge_reasoning": judge_evaluation.get("reason")
    })

# DataFrame Creation and Export
df = pd.DataFrame(rows)

# Renaming prediction column to the specific model name
df.rename(columns={'model_prediction': 'Qwen'}, inplace=True)

output_metrics_path = "/content/drive/MyDrive/results_my_dataset_qwen_screenqa_json/results_my_dataset_evaluated_GPT_qwen_metrics.json"

df.to_json(output_metrics_path, orient='records', indent=4, force_ascii=False)

print(f"File successfully saved to: {output_metrics_path}")

In [ ]:
# Performance Metrics Calculation
success_threshold = 7
average_judge_score = df["gpt_judge_score"].mean()
num_successes = (df["gpt_judge_score"] >= success_threshold).sum()
total_items = len(df)
judge_accuracy = (num_successes / total_items) * 100

print("\n" + "="*40)
print("       PERFORMANCE REPORT")
print("="*40)
print(f"Mean SQuAD F1:             {df['squad_f1'].mean() * 100:.2f}%")
print(f"Mean SQuAD EM:             {df['squad_em'].mean() * 100:.2f}%")
print(f"Mean Cosine Similarity:    {df['semantic_similarity'].mean() * 100:.2f}%")
print(f"Mean Judge Score (0-10):   {average_judge_score:.2f}")
print(f"Accuracy (Threshold >= {success_threshold}):  {judge_accuracy:.2f}%")
print("="*40)

df.head()

In [ ]:
df.describe()